In [1]:
pip install faiss-cpu

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
    --------------------------------------- 0.3/16.5 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/16.5 MB 6.4 MB/s eta 0:00:03
   ------ --------------------------------- 2.9/16.5 MB 7.0 MB/s eta 0:00:02
   ----------- ---------------------------- 4.7/16.5 MB 6.6 MB/s eta 0:00:02
   ---------------- ----------------------- 6.8/16.5 MB 7.4 MB/s eta 0:00:02
   ---------------------- ----------------- 9.2/16.5 MB 8.3 MB/s eta 0:00:01
   ----------------------------- ---------- 12.1/16.5 MB 9.1 MB/s eta 0:00:01
   ------------------------------------ --- 15.2/16.5 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 16.5/16.5 MB 10.1 MB/s  0:00:01
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------- ----------------------------- 3.4/12.6 MB 16.3 MB/s eta 0:00:01
   -------------------- ------------------- 6.6/12.6 MB 16.0 MB/s eta 0:00:01
   -------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import faiss

In [2]:
faiss.__version__

'1.15.0'

In [3]:
milestone_text = """"

Sudhanshu's commitment to affordable education wasn't just a business strategy—it was his life's mission. Over the years, iNeuron has helped over 1.5 million students from 34+ countries, providing them with the skills they need to succeed in today's competitive job market. Many of these students, like Sudhanshu himself, came from disadvantaged backgrounds. They saw iNeuron as a lifeline—an opportunity to rise above their circumstances.

In 2022, iNeuron was acquired by PhysicsWallah in a deal worth ₹250 crore. While this acquisition was a significant milestone, Sudhanshu remained focused on his mission. Even after the acquisition, iNeuron continued to offer some of the most affordable and accessible tech courses in the world.

"""

In [4]:
import requests
import os
import json
import numpy as np

In [5]:
def chunk_text(text,chunk_size=20, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [6]:
chunks = chunk_text(milestone_text)

In [7]:
chunks

['" Sudhanshu\'s commitment to affordable education wasn\'t just a business strategy—it was his life\'s mission. Over the years, iNeuron has',
 "strategy—it was his life's mission. Over the years, iNeuron has helped over 1.5 million students from 34+ countries, providing them",
 "helped over 1.5 million students from 34+ countries, providing them with the skills they need to succeed in today's competitive",
 "with the skills they need to succeed in today's competitive job market. Many of these students, like Sudhanshu himself, came",
 'job market. Many of these students, like Sudhanshu himself, came from disadvantaged backgrounds. They saw iNeuron as a lifeline—an opportunity',
 'from disadvantaged backgrounds. They saw iNeuron as a lifeline—an opportunity to rise above their circumstances. In 2022, iNeuron was acquired',
 'to rise above their circumstances. In 2022, iNeuron was acquired by PhysicsWallah in a deal worth ₹250 crore. While this',
 'by PhysicsWallah in a deal worth ₹250 c

In [8]:
API_URL = "https://api.euron.one/api/v1/euri/embeddings"
API_KEY = "euri-9d7448e4925a8a793a0630e20659cf0f4f80cd25fb7b8b1bafcbd87af0f79c34"
MODEL_NAME = "text-embedding-3-small"


In [11]:
header = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

In [14]:

all_embeddings = []


for i, chunk in enumerate(chunks):
    payload = {
        "model": MODEL_NAME,
        "input": chunk
    }
    response = requests.post(API_URL, headers= header, data= json.dumps(payload))
    data = response.json()
    embeddings = data['data'][0]['embedding']
    all_embeddings.append(embeddings)


In [10]:
for i in enumerate(chunks):
    print(i)

(0, '" Sudhanshu\'s commitment to affordable education wasn\'t just a business strategy—it was his life\'s mission. Over the years, iNeuron has')
(1, "strategy—it was his life's mission. Over the years, iNeuron has helped over 1.5 million students from 34+ countries, providing them")
(2, "helped over 1.5 million students from 34+ countries, providing them with the skills they need to succeed in today's competitive")
(3, "with the skills they need to succeed in today's competitive job market. Many of these students, like Sudhanshu himself, came")
(4, 'job market. Many of these students, like Sudhanshu himself, came from disadvantaged backgrounds. They saw iNeuron as a lifeline—an opportunity')
(5, 'from disadvantaged backgrounds. They saw iNeuron as a lifeline—an opportunity to rise above their circumstances. In 2022, iNeuron was acquired')
(6, 'to rise above their circumstances. In 2022, iNeuron was acquired by PhysicsWallah in a deal worth ₹250 crore. While this')
(7, 'by PhysicsWalla

In [16]:
len(all_embeddings)

12

In [17]:
type(all_embeddings)

list

In [18]:
embedding_array = np.array(all_embeddings, dtype= 'float32')

In [19]:
embedding_array

array([[-0.01930237,  0.00639725,  0.0176239 , ..., -0.00151825,
        -0.00459671, -0.01124573],
       [ 0.0016737 ,  0.01316071,  0.02632141, ..., -0.02128601,
        -0.02079773, -0.037323  ],
       [ 0.01687622, -0.02314758,  0.04595947, ..., -0.02998352,
        -0.02294922, -0.0020504 ],
       ...,
       [-0.02407837, -0.01535797,  0.04220581, ..., -0.01493073,
         0.00017381,  0.02323914],
       [-0.05905151,  0.0166626 , -0.02613831, ..., -0.02272034,
         0.01792908,  0.02067566],
       [ 0.00730515, -0.01882935, -0.01434326, ..., -0.03616333,
         0.02223206, -0.00165462]], shape=(12, 1536), dtype=float32)

In [20]:
base_index = faiss.IndexFlatL2(embedding_array.shape[1])

In [21]:
base_index.add(embedding_array)

In [22]:
faiss.write_index(base_index,"faiss_index.faiss")

In [33]:
search_text = "what is the mission of sudhanshu"

In [34]:

def embedding_text(text):
    payload = {
        "model": MODEL_NAME,
        "input": text
    }
    response = requests.post(API_URL, headers= header, data= json.dumps(payload))
    data = response.json()
    embeddings = data['data'][0]['embedding']
    emb = np.array(embeddings, dtype= 'float32').reshape(1,-1)
    return emb
    

In [35]:
query_text_emb = embedding_text(search_text)

In [36]:
base_index.search(query_text_emb,3)

(array([[0.9569905, 1.1366327, 1.1689651]], dtype=float32), array([[0, 3, 8]]))

In [32]:
chunks[0]

'" Sudhanshu\'s commitment to affordable education wasn\'t just a business strategy—it was his life\'s mission. Over the years, iNeuron has'